# Generative AI Assistant for Knowledge Work

## Project Overview

This project is a Google Colab-based prototype inspired by Microsoft Copilot and generative AI tools for knowledge workers. The system allows users to upload a research article or business document, extract text, summarize the content, identify important keywords, ask questions from the document, and generate a structured project report.

## Problem Statement

Knowledge workers spend significant time reading long documents, extracting insights, preparing reports, and answering document-specific questions. This project demonstrates how generative AI and NLP models can reduce manual effort and improve productivity.

## Objective

The objective of this project is to build a free, no-API-key AI assistant using Hugging Face models that can support document understanding and workplace productivity tasks.

## System Architecture

User uploads PDF  
↓  
Python extracts text using pypdf  
↓  
Hugging Face summarization model creates summary  
↓  
TF-IDF extracts important keywords  
↓  
Question-answering model answers document-based questions  
↓  
System generates downloadable analysis report  

## Technologies Used

- Google Colab
- Python
- Hugging Face Transformers
- BART Summarization Model
- RoBERTa Question Answering Model
- pypdf
- scikit-learn TF-IDF

# Document Upload

In [ ]:
from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print("Uploaded file:", file_name)

Saving Machine Learning Healthcare.pdf to Machine Learning Healthcare (3).pdf
Uploaded file: Machine Learning Healthcare (3).pdf


#Extract Text from PDF

In [ ]:
from pypdf import PdfReader

def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text()

        if page_text:
            text += f"\n--- Page {page_number} ---\n"
            text += page_text + "\n"

    return text

document_text = extract_text_from_pdf(file_name)

print("Text extracted successfully.")
print("Total characters:", len(document_text))
print("PDF text is stored in the variable: document_text")
print("Preview hidden for copyright-safe notebook sharing.")

Text extracted successfully.
Total characters: 27580
PDF text is stored in the variable: document_text
Preview hidden for copyright-safe notebook sharing.


#Load Hugging Face summarization model

# BART Summarization Model

This project uses `facebook/bart-large-cnn`, a BART-based transformer model from Hugging Face, to summarize extracted PDF text. Since long documents exceed the model input limit, the text is divided into smaller chunks, summarized separately, and then combined into a final summary.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
hf_model = hf_model.to(device)

print("Model loaded on:", device)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded on: cpu


#Summarize the PDF

In [ ]:
def summarize_chunk(chunk, max_length=180, min_length=60):
    inputs = tokenizer(
        chunk,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = hf_model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


def summarize_long_text(text, chunk_size=2500, max_chunks=5):
    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]

        if len(chunk.strip()) > 500:
            chunks.append(chunk)

    summaries = []

    total_chunks = min(len(chunks), max_chunks)

    for idx, chunk in enumerate(chunks[:max_chunks], start=1):
        print(f"Summarizing chunk {idx}/{total_chunks}...")
        summaries.append(summarize_chunk(chunk))

    return "\n\n".join(summaries)


summary = summarize_long_text(document_text)

print("Chunk-level summary generated successfully.")
print("Summary is stored in the variable: summary")

Summarizing chunk 1/5...
Summarizing chunk 2/5...
Summarizing chunk 3/5...
Summarizing chunk 4/5...
Summarizing chunk 5/5...
Chunk-level summary generated successfully.
Summary is stored in the variable: summary


# Create final clean summary

In [ ]:
final_summary = summarize_chunk(
    summary,
    max_length=220,
    min_length=80
)

print("Final summary generated successfully.")
print("Final summary is stored in the variable: final_summary")

Final summary generated successfully.
Final summary is stored in the variable: final_summary


In [ ]:
print(final_summary)

Machine Learning is a field of study that gives the computer the ability to learn without being explicitlyprogrammed. Machine learning algorithms are broadly categorized into four parts: Supervised, Unsupervised, Semi-supervised and Decision Tree. Computer vision is a broad segment which enables the machine to implement machine learning. This paper explores the work done in the healthcare sector with the help of machine learning and deep learning techniques.


#Extract Keywords

In [ ]:
!pip install -q scikit-learn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

def extract_keywords_tfidf(text, top_n=15):
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=1000,
        ngram_range=(1, 2)
    )

    tfidf_matrix = vectorizer.fit_transform([text])
    scores = tfidf_matrix.toarray()[0]
    feature_names = vectorizer.get_feature_names_out()

    keyword_scores = list(zip(feature_names, scores))
    keyword_scores = sorted(keyword_scores, key=lambda x: x[1], reverse=True)

    return keyword_scores[:top_n]

keywords = extract_keywords_tfidf(document_text)

print("TOP KEYWORDS:\n")
for keyword, score in keywords:
    print(f"- {keyword}: {score:.4f}")

TOP KEYWORDS:

- learning: 0.4506
- machine: 0.3836
- machine learning: 0.3390
- data: 0.1561
- accuracy: 0.1517
- al: 0.1472
- et: 0.1427
- et al: 0.1427
- healthcare: 0.1338
- disease: 0.1294
- techniques: 0.1115
- cancer: 0.1026
- heart: 0.0981
- algorithms: 0.0937
- diseases: 0.0937


# Save Report as TXT File

In [ ]:
with open("knowledge_work_ai_report.txt", "w", encoding="utf-8") as file:
    file.write(project_report)

print("Report saved as knowledge_work_ai_report.txt")

Report saved as knowledge_work_ai_report.txt


In [ ]:
from google.colab import files

files.download("knowledge_work_ai_report.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Load Free Question-Answering Model

# Document Question Answering

This project uses `deepset/roberta-base-squad2`, a RoBERTa-based question-answering model, to answer questions from the uploaded document. The model extracts answers from the document context rather than generating unsupported responses.

In [ ]:
from transformers import pipeline

qa_model = pipeline(
    task="question-answering",
    model="deepset/roberta-base-squad2"
)

print("Question-answering model loaded successfully.")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Question-answering model loaded successfully.


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


# Create Question-Answering Function

In [ ]:
def ask_question_from_document(question, document_text, context_size=4000):
    context = document_text[:context_size]

    result = qa_model(
        question=question,
        context=context
    )

    return result

# Ask your First Question

In [ ]:
question = "What is the main topic of this document?"

answer = ask_question_from_document(question, document_text)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer["answer"])

print("\nCONFIDENCE SCORE:")
print(answer["score"])

QUESTION:
What is the main topic of this document?

ANSWER:
data science in healthcare

CONFIDENCE SCORE:
0.13661165535449982


#Ask Custom Questions

In [ ]:
user_question = input("Ask a question about the document: ")

answer = ask_question_from_document(user_question, document_text)

print("\nANSWER:")
print(answer["answer"])

print("\nCONFIDENCE SCORE:")
print(answer["score"])

Ask a question about the document: what features they using

ANSWER:
representation, evaluation 
and optimization

CONFIDENCE SCORE:
0.20941312611103058


# Add Q&A Results

In [ ]:
sample_questions = [
    "What is the main topic of this document?",
    "What is the role of machine learning in healthcare?",
    "What are the benefits of machine learning?",
    "What challenges are mentioned in the paper?"
]

qa_results = []

for q in sample_questions:
    result = ask_question_from_document(q, document_text)
    qa_results.append((q, result["answer"], result["score"]))

qa_section = "\n6. SAMPLE QUESTION ANSWERING\n\n"

for q, ans, score in qa_results:
    qa_section += f"Question: {q}\n"
    qa_section += f"Answer: {ans}\n"
    qa_section += f"Confidence Score: {score:.4f}\n\n"

project_report_with_qa = project_report + qa_section

print(project_report_with_qa)


GENERATIVE AI ASSISTANT FOR KNOWLEDGE WORK
Research Article Analysis Report

1. DOCUMENT SUMMARY

Machine Learning is a field of study that gives the computer the ability to learn without being explicitlyprogrammed. Machine learning algorithms are broadly categorized into four parts: Supervised, Unsupervised, Semi-supervised and Decision Tree. Computer vision is a broad segment which enables the machine to implement machine learning. This paper explores the work done in the healthcare sector with the help of machine learning and deep learning techniques.

2. IMPORTANT KEYWORDS

learning, machine, machine learning, data, accuracy, al, et, et al, healthcare, disease, techniques, cancer, heart, algorithms, diseases

3. BUSINESS VALUE

This prototype helps knowledge workers quickly understand long documents by automatically extracting text from PDFs, summarizing important content, and identifying major keywords. It reduces manual reading time and supports faster research, reporting, and d

# Save Updated Report

In [ ]:
with open("knowledge_work_ai_report_with_qa.txt", "w", encoding="utf-8") as file:
    file.write(project_report_with_qa)

print("Updated report saved as knowledge_work_ai_report_with_qa.txt")

Updated report saved as knowledge_work_ai_report_with_qa.txt


#Download Updated Report

In [ ]:
from google.colab import files

files.download("knowledge_work_ai_report_with_qa.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Creating Copilot Style Menu Function

In [ ]:
def knowledge_work_copilot():
    print("=" * 60)
    print("GENERATIVE AI ASSISTANT FOR KNOWLEDGE WORK")
    print("=" * 60)
    print("Choose what you want to do:")
    print("1. Summarize document")
    print("2. Show important keywords")
    print("3. Ask a question about the document")
    print("4. Generate project report")
    print("5. Save and download report")
    print("6. Exit")
    print("=" * 60)

    choice = input("Enter your choice from 1 to 6: ")

    if choice == "1":
        print("\nDOCUMENT SUMMARY:\n")
        print(final_summary)

    elif choice == "2":
        print("\nIMPORTANT KEYWORDS:\n")
        for keyword, score in keywords:
            print(f"- {keyword}: {score:.4f}")

    elif choice == "3":
        question = input("\nAsk a question about the document: ")
        answer = ask_question_from_document(question, document_text)

        print("\nANSWER:")
        print(answer["answer"])

        print("\nCONFIDENCE SCORE:")
        print(answer["score"])

    elif choice == "4":
        print("\nPROJECT REPORT:\n")
        print(project_report_with_qa)

    elif choice == "5":
        with open("knowledge_work_copilot_report.txt", "w", encoding="utf-8") as file:
            file.write(project_report_with_qa)

        from google.colab import files
        files.download("knowledge_work_copilot_report.txt")

        print("Report downloaded successfully.")

    elif choice == "6":
        print("Exiting Knowledge Work Copilot.")

    else:
        print("Invalid choice. Please enter a number from 1 to 6.")

#Run the Copilot Menu

In [ ]:
knowledge_work_copilot()

GENERATIVE AI ASSISTANT FOR KNOWLEDGE WORK
Choose what you want to do:
1. Summarize document
2. Show important keywords
3. Ask a question about the document
4. Generate project report
5. Save and download report
6. Exit
Enter your choice from 1 to 6: 1

DOCUMENT SUMMARY:

Machine Learning is a field of study that gives the computer the ability to learn without being explicitlyprogrammed. Machine learning algorithms are broadly categorized into four parts: Supervised, Unsupervised, Semi-supervised and Decision Tree. Computer vision is a broad segment which enables the machine to implement machine learning. This paper explores the work done in the healthcare sector with the help of machine learning and deep learning techniques.


## Conclusion

This project demonstrates a practical prototype of a Copilot-style knowledge work assistant. It helps users analyze long documents by automatically generating summaries, extracting keywords, answering questions, and creating structured reports.

Although this version uses free local models instead of paid APIs, it successfully demonstrates how generative AI can support productivity, research, and decision-making in knowledge work.

## Limitations

- The model may not capture every detail from long documents.
- The question-answering system uses a limited context window.
- Summary quality may be lower than commercial LLMs.
- The current version runs in Google Colab and is not yet deployed as a web application.

## Future Scope

- Deploy the project using Streamlit.
- Add a web-based file upload interface.
- Improve long-document search using embeddings and retrieval.
- Add email drafting and slide-outline generation.
- Integrate with Microsoft 365, SharePoint, or OneDrive in a future enterprise version.